<h1>Chapter 2 - Generation Models</h1>
<i>Choosing the generation model for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch01_RAG_intro/rag_basics.ipynb)

---

This notebook is for Chapter 3 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [46]:
!pip install openai anthropic

### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [13]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [5]:
!git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
%cd RAG-with-Python-Cookbook
!git sparse-checkout init --cone
!git sparse-checkout set datasets
!git checkout
!cp -r datasets /content/datasets


Cloning into 'RAG-with-Python-Cookbook'...
remote: Enumerating objects: 1250, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 1250 (delta 185), reused 156 (delta 153), pack-reused 1014 (from 1)
Receiving objects: 100% (1250/1250), 41.29 MiB | 10.68 MiB/s, done.
Resolving deltas: 100% (721/721), done.
/content/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.


## 1. OpenAI Chat Completions

In [6]:
from openai import OpenAI

def ask_with_context(context, question):
    client = OpenAI()

    messages = [
        {
            "role": "system",
            "content": "Answer based only on the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        },
    ]

    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=messages
    )

    return response.choices[0].message.content


# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

RAG stands for **Retrieval-Augmented Generation**.


## 2. OpenAI Whisper Speech-to-Text

In [8]:
import httpx
from openai import OpenAI

client = OpenAI(http_client=httpx.Client(verify=False))

with open(
    "./datasets/audio_files/harvard.wav",
    "rb",
) as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
    )

print(transcript.text)

The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.


## 3. Anthropic Claude Example

In [10]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in "
                       "simple terms.",
        }
    ],
)

print(response.content[0].text)

# Vector Databases Explained Simply

## The Core Problem They Solve

Traditional databases find exact matches ("find user where id = 42"). But sometimes you need to find **similar** things, not exact things — like "find images similar to this photo" or "find documents related to this question."

---

## The Key Concept: Vectors (Embeddings)

Think of a vector as a **list of numbers that describes something**.

```
"Dog"   → [0.2, 0.8, 0.1, 0.9, ...]
"Cat"   → [0.3, 0.7, 0.2, 0.8, ...]
"Car"   → [0.9, 0.1, 0.8, 0.1, ...]
```

- Similar things produce **similar number lists**
- An AI model converts your data


## 4. Gemini API Example using the OpenAI SDK

In [24]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(resp.choices[0].message.content)

The capital of France is **Paris**.


## 5. Deploy local LLMs using Ollama

In [32]:
"""
Running Local LLMs with Ollama
Shows how to use locally hosted models via Ollama with the OpenAI SDK

pip install openai
"""

from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama does not require a real key,
                       # but the SDK expects one
)

response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is retrieval augmented generation?"
        },
    ],
)

print(response.choices[0].message.content)

APIConnectionError: Connection error.

In [33]:
from openai import OpenAI

models = ["llama2", "mistral", "codellama"]
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

for model in models:
    print(f"\n--- Testing {model} ---")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": "Explain RAG in one sentence."}
        ]
    )
    print(response.choices[0].message.content)


--- Testing llama2 ---


APIConnectionError: Connection error.

## 6. Pydantic Structured Output

In [55]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import date
from typing import List

class LineItem(BaseModel):
    description: str
    quantity: int
    total: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    supplier: str
    items: List[LineItem]
    total_due: float

client = OpenAI()

response = client.responses.parse(
    model="gpt-5",
    input=[
        {
            "role": "system",
            "content": "Extract invoice information into the schema.",
        },
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Extract invoice data."},
                {
                    "type": "input_image",
                    "image_url": "https://example.com/invoice.png",
                },
            ],
        },
    ],
    text_format=Invoice,  # ← matches docs
)

invoice = response.output_parsed

print(invoice)

BadRequestError: Error code: 400 - {'error': {'message': 'Error while downloading https://example.com/invoice.png. Upstream status code: 404.', 'type': 'invalid_request_error', 'param': 'url', 'code': 'invalid_value'}}